# 🤟 BISINDO CSLR — Backend Server (Google Colab + GPU)

Notebook ini menjalankan **backend FastAPI** pipeline CSLR BISINDO di Google Colab dengan GPU T4.

Setelah server berjalan, Anda akan mendapatkan **URL publik** (via ngrok) yang dapat digunakan sebagai target backend di frontend lokal.

---

### Prasyarat
- Runtime: **GPU** (Runtime → Change runtime type → T4 GPU)
- Akun [ngrok](https://ngrok.com) gratis (untuk mendapat authtoken)

---

### Urutan Eksekusi
Jalankan sel **secara berurutan** dari atas ke bawah.

## ✅ Step 0 — Verifikasi GPU

In [1]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU tidak terdeteksi. Pastikan runtime sudah diset ke GPU.")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


## 📦 Step 1 — Clone Repository

In [2]:
import os

REPO_DIR = "/content/bisindo-cslr"
REPO_URL = "https://github.com/MahardikaPratama/bisindo-cslr.git"

if os.path.exists(REPO_DIR):
    print("Repo sudah ada, skip clone.")
else:
    !git clone --recursive "{REPO_URL}" "{REPO_DIR}"
    print("✅ Clone selesai.")

%cd {REPO_DIR}
!ls -la

Cloning into '/content/bisindo-cslr'...
remote: Enumerating objects: 300, done.
remote: Counting objects: 100% (300/300), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 300 (delta 81), reused 283 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (300/300), 582.34 KiB | 20.08 MiB/s, done.
Resolving deltas: 100% (81/81), done.
✅ Clone selesai.
/content/bisindo-cslr
total 128
drwxr-xr-x 7 root root  4096 May 24 17:19 .
drwxr-xr-x 1 root root  4096 May 24 17:19 ..
-rw-r--r-- 1 root root 12774 May 24 17:19 app.py
-rw-r--r-- 1 root root  9798 May 24 17:19 bisindo_cslr_colab.ipynb
-rw-r--r-- 1 root root  5213 May 24 17:19 CODING_STANDARDS.md
-rw-r--r-- 1 root root   297 May 24 17:19 environment.yml
drwxr-xr-x 8 root root  4096 May 24 17:19 .git
-rw-r--r-- 1 root root   670 May 24 17:19 .gitignore
drwxr-xr-x 2 root root  4096 May 24 17:19 inference
-rw-r--r-- 1 root root  1074 May 24 17:19 LICENSE
-rw-r--r-- 1 root root  6617 May 24 17:19 main.py
drwxr-xr-x 9 r

In [12]:
%cd /content/bisindo-cslr/mslr_iccv2025
!git clone --recursive https://github.com/WayenVan/ctcdecode.git
%cd ctcdecode
!pip install .
%cd /content/bisindo-cslr

/content/bisindo-cslr/mslr_iccv2025
Cloning into 'ctcdecode'...
remote: Enumerating objects: 1400, done.
remote: Counting objects: 100% (367/367), done.
remote: Compressing objects: 100% (324/324), done.
remote: Total 1400 (delta 38), reused 358 (delta 35), pack-reused 1033 (from 1)
Receiving objects: 100% (1400/1400), 1.22 MiB | 16.64 MiB/s, done.
Resolving deltas: 100% (532/532), done.
/content/bisindo-cslr/mslr_iccv2025/ctcdecode
Processing /content/bisindo-cslr/mslr_iccv2025/ctcdecode
  Preparing metadata (setup.py) ... done
  Created wheel for ctcdecode: filename=ctcdecode-1.0.3-cp312-cp312-linux_x86_64.whl size=19609620 sha256=e0ff4833681722cf80bd6c73ee9b9f8b61abfaaeb8dea85921660c2393ccdcd0
  Stored in directory: /tmp/pip-ephem-wheel-cache-qmdh24h6/wheels/49/6d/b6/6b467f96e2cdd7139f3c89e8b3d3ed0ed2459f9d5c222d439f
Successfully built ctcdecode
/content/bisindo-cslr


## 🐍 Step 2 — Install Dependencies

In [4]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Bersihkan dependency lama
!pip uninstall -y -q \
    numpy \
    protobuf \
    opencv-python \
    opencv-python-headless \
    mediapipe \
    tensorflow \
    tensorflow-cpu > /dev/null 2>&1

# Install versi compatible
!pip install -q \
    --disable-pip-version-check \
    --no-warn-conflicts \
    numpy==1.26.4 \
    protobuf==4.25.3 \
    mediapipe==0.10.14 \
    opencv-python==4.10.0.84 \
    matplotlib==3.8.4 \
    fastapi==0.111.0 \
    uvicorn==0.30.1 \
    python-multipart==0.0.9 \
    scipy \
    pyyaml \
    tqdm \
    gdown \
    pyngrok

print("✅ Dependencies compatible installed.")

✅ Dependencies compatible installed.


In [5]:
import numpy as np
import mediapipe as mp
import cv2
import google.protobuf

print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)
print("OpenCV:", cv2.__version__)
print("Protobuf:", google.protobuf.__version__)

print("\n✅ Core dependency import berhasil.")

NumPy: 2.0.2
MediaPipe: 0.10.14
OpenCV: 4.10.0
Protobuf: 4.25.3

✅ Core dependency import berhasil.


## 🔽 Step 3 — Download Model dari Google Drive

In [6]:
import gdown
import os

MODEL_DIR  = "/content/bisindo-cslr/mslr_iccv2025/model"
MODEL_PATH = os.path.join(MODEL_DIR, "best_dev_01.30_epoch39_model.pt")
GDRIVE_ID  = "1Uw6nJnR74DtNp3xhGi5kCT702I8II_As"

os.makedirs(MODEL_DIR, exist_ok=True)

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"✅ Model sudah ada ({size_mb:.0f} MB), skip download.")
else:
    print("⬇️  Mengunduh model dari Google Drive (~680 MB)...")
    url = f"https://drive.google.com/uc?id={GDRIVE_ID}"
    gdown.download(url, MODEL_PATH, quiet=False)
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"✅ Download selesai ({size_mb:.0f} MB) → {MODEL_PATH}")

⬇️  Mengunduh model dari Google Drive (~680 MB)...


Downloading...
From (original): https://drive.google.com/uc?id=1Uw6nJnR74DtNp3xhGi5kCT702I8II_As
From (redirected): https://drive.google.com/uc?id=1Uw6nJnR74DtNp3xhGi5kCT702I8II_As&confirm=t&uuid=750829b6-2a5b-4784-9059-3f8e717e03cb
To: /content/bisindo-cslr/mslr_iccv2025/model/best_dev_01.30_epoch39_model.pt
100%|██████████| 712M/712M [00:08<00:00, 87.5MB/s]

✅ Download selesai (712 MB) → /content/bisindo-cslr/mslr_iccv2025/model/best_dev_01.30_epoch39_model.pt


## 🔑 Step 4 — Konfigurasi ngrok Authtoken

Daftar akun gratis di [https://ngrok.com](https://ngrok.com) → **Your Authtoken** → copy dan paste di bawah.

In [7]:
from pyngrok import ngrok

NGROK_TOKEN = "3DXuk4IKY5lCWVt2VZo5X5fLCER_6ALsKCfwqfR8dJn7VMcNE"

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok authtoken dikonfigurasi.")

✅ ngrok authtoken dikonfigurasi.


## 🚀 Step 5 — Jalankan Backend Server

> **Catatan:** Backend di `app.py` sekarang memakai config default hasil training `Baseline+TN.yaml` ketika frontend tidak mengirim `config_name`. Sel ini berjalan secara **background**. Tunggu sampai pesan `✅ Server siap` muncul sebelum lanjut ke Step 6.

In [22]:
import subprocess
import time
import requests
import os

os.chdir("/content/bisindo-cslr")

# Jalankan uvicorn di background
server_process = subprocess.Popen(
    [
        "uvicorn", "app:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--log-level", "info"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd="/content/bisindo-cslr"
)

print("⏳ Menunggu server startup...")
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("✅ Server siap di http://127.0.0.1:8000")
            break
    except Exception:
        pass
else:
    print("❌ Server gagal start dalam 30 detik. Cek log di bawah.")
    out, _ = server_process.communicate(timeout=2)
    print(out.decode())

⏳ Menunggu server startup...
✅ Server siap di http://127.0.0.1:8000


## 🌐 Step 6 — Buat Tunnel ngrok (URL Publik)

In [23]:
from pyngrok import ngrok

# Buat tunnel HTTP ke port 8000
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print("=" * 60)
print("✅ BACKEND SIAP!")
print("=" * 60)
print(f"\n🔗 Public URL  : {PUBLIC_URL}")
print(f"📡 Health check: {PUBLIC_URL}/health")
print(f"📖 API Docs    : {PUBLIC_URL}/docs")
print("\n" + "=" * 60)
print("📋 LANGKAH SELANJUTNYA:")
print("   Salin URL di atas, lalu update vite.config.ts di frontend:")
print("")
print("   proxy: {")
print(f"     '/api'    : {{ target: '{PUBLIC_URL}', changeOrigin: true }},")
print(f"     '/preview': {{ target: '{PUBLIC_URL}', changeOrigin: true }},")
print("   }")
print("=" * 60)

✅ BACKEND SIAP!

🔗 Public URL  : https://fridge-catchy-carving.ngrok-free.dev
📡 Health check: https://fridge-catchy-carving.ngrok-free.dev/health
📖 API Docs    : https://fridge-catchy-carving.ngrok-free.dev/docs

📋 LANGKAH SELANJUTNYA:
   Salin URL di atas, lalu update vite.config.ts di frontend:

   proxy: {
     '/api'    : { target: 'https://fridge-catchy-carving.ngrok-free.dev', changeOrigin: true },
     '/preview': { target: 'https://fridge-catchy-carving.ngrok-free.dev', changeOrigin: true },
   }


## 🔍 Step 7 — Verifikasi Endpoint (Opsional)

In [34]:
import requests
import json

# Test /health
r = requests.get(f"{PUBLIC_URL}/health")
print(f"/health → {r.status_code}: {r.json()}")

print("\n✅ Backend berjalan dan dapat diakses dari luar Colab.")
print("\n⚠️  PENTING: Jangan tutup tab Colab ini selama frontend digunakan.")
print("   Sesi ngrok akan mati jika notebook di-disconnect.")

/health → 200: {'status': 'ok'}

✅ Backend berjalan dan dapat diakses dari luar Colab.

⚠️  PENTING: Jangan tutup tab Colab ini selama frontend digunakan.
   Sesi ngrok akan mati jika notebook di-disconnect.


## 📊 Step 8 — Monitor Log Server (Opsional)

Jalankan sel ini untuk melihat log server secara real-time.

> Tekan **stop (■)** di sebelah sel untuk menghentikan streaming log.

In [35]:
import sys

print("📋 Streaming server log (Ctrl+C atau stop untuk berhenti)...\n")
try:
    for line in iter(server_process.stdout.readline, b''):
        decoded = line.decode('utf-8', errors='replace').rstrip()
        print(decoded, flush=True)
except KeyboardInterrupt:
    print("\n⏹️  Log streaming dihentikan.")

📋 Streaming server log (Ctrl+C atau stop untuk berhenti)...



AttributeError: 'str' object has no attribute 'decode'

## 🛑 Step 9 — Hentikan Server (Opsional)

Jalankan hanya jika ingin menghentikan server sebelum sesi Colab berakhir.

In [ ]:
from pyngrok import ngrok as _ngrok

# Tutup tunnel ngrok
_ngrok.kill()

# Hentikan uvicorn
if 'server_process' in dir():
    server_process.terminate()
    server_process.wait()

print("✅ Server dan tunnel ngrok dihentikan.")